# GIS interoperability and export (shapefile, raster)
This notebook shows how to export gridmarthe features to common GIS format.


## Script API

In [12]:
# import modules
import numpy as np
import gridmarthe as gm

### Shapefile, step 1: conversion

``gridmarthe`` use the ``geopandas`` API to convert grids to a geodataframe object.

In [ ]:
path_to_data = './data'
ds = gm.load_marthe_grid(f'{path_to_data}/chasim_hallue.out', 'CHARGE', fpastp=f'{path_to_data}/hallue.pastp').isel(time=-1)
gdf = gm.to_geodataframe(ds)
gdf.head()

/mnt/d/home/gridmarthe/src/gridmarthe/operasem/gis.py:65: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  if 'time' in ds.dims.keys():


,charge,x,y,dx,dy,time,geometry
zone,,,,,,,
1,9999.0,596.75,2568.75,0.5,0.5,2012-07-01,"POLYGON ((596.5 2568.5, 597 2568.5, 597 2569, ..."
2,9999.0,597.25,2568.75,0.5,0.5,2012-07-01,"POLYGON ((597 2568.5, 597.5 2568.5, 597.5 2569..."
3,9999.0,597.75,2568.75,0.5,0.5,2012-07-01,"POLYGON ((597.5 2568.5, 598 2568.5, 598 2569, ..."
4,9999.0,598.25,2568.75,0.5,0.5,2012-07-01,"POLYGON ((598 2568.5, 598.5 2568.5, 598.5 2569..."
5,9999.0,598.75,2568.75,0.5,0.5,2012-07-01,"POLYGON ((598.5 2568.5, 599 2568.5, 599 2569, ..."


### Shapefile, step 2: export

In [ ]:
# gdf.to_file('chasim.shp')

### Export to raster

For one time step, user can export a regular grid as a TIFF file with function ``glm.to_raster``
For all time steps at once, use ``ds.mart.to_raster(filename_tpl='chasim_hallue')``, the filename template
argument must not contain any extension.

**CAUTION**: this is only for regular grids. For irregular/nested grids,
the QGIS plugin PostMARTHE is preferable to save chasim as .tiff
It will save the results in raster file with the smallest resolution
available in all nested grids.

In [ ]:
help(gm.to_raster)  # function doc to write a raster file

Help on function to_raster in module gridmarthe.operasem.gis:

to_raster(da, x_dim='x', y_dim='y', epsg=27572, fout='raster.tiff')
    Write a xr.DataArray to a raster file
    
    need xarray with rioxarray installed.
    Only for regular grids.
    TODO: add support for irregular grids, using PostMARTHE QGIS plugin code.
    
    Parameters
    ----------
    da : xarray.DataArray
        The DataArray to write to a raster file (only 2D, i.e. no time dimension, select
        a variable and timestep before).
    x_dim : str, optional
        The name of the x dimension, by default 'x'.
    y_dim : str, optional
        The name of the y dimension, by default 'y'.
    epsg : int, optional
        The EPSG code for the coordinate reference system, by default 27572.
    fout : str, optional
        The output file path for the raster file, by default 'raster.tiff'.
    
    Returns
    -------
    None
        The function writes the raster file and returns None.



## CLI for gis transformation

A command line script is also provided to transform a marthe grid file to shapefile or geopackage.

In [15]:
!martshp -h

usage: martshp [-h] [--output OUTPUT] [--variable VARIABLE]
               [--xyfactor XYFACTOR] [--gpkg] [--mask] [--version]
               [--wide_fmt]
               [grid timesteps ...]

Convert a Marthe GridFile to shapefile format.

positional arguments:
  grid timesteps        Paths to grid [and timesteps if result] files are
                        expected

options:
  -h, --help            show this help message and exit
  --output OUTPUT, -o OUTPUT
                        Output filename. Default is input.nc
  --variable VARIABLE, -v VARIABLE
                        Variable (field) to read, default is None: i.e
                        variable will be parsed from file and ONLY the first
                        variable will be read. Pass 'all' to get all
                        variables.
  --xyfactor XYFACTOR, -x XYFACTOR
                        Transformation factor for coordinates. Optionnal,
                        default is 1 (no transformation).
  --gpkg, -g         